In [ ]:
#editado 10 11 - 15 24
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import openpyxl
from selenium.webdriver.chrome.options import Options
import os
from selenium.webdriver.support.ui import Select
from datetime import datetime
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from pathlib import Path
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains



################## CONFIGURAÇÕES PREENCHER ################

##EXCEL
caminho_planilha = r"Y:\DOS_GOC_SCM\GOB\AUTOMAÇÃO - NÃO DELETAR\PLANILHA EXECUCAO INSERIR AP.xlsx"
planilha = openpyxl.load_workbook(caminho_planilha)
aba = planilha["TB MATEUS REMOTO 1"]
LINHA_INICIAL = 2 #linha inicial de execução do excel


#PASTAS COMPUTADOR
PASTA_NFS = Path(r"Y:\DOS_GOC_SCM\GOB\AUTOMAÇÃO - NÃO DELETAR\FILA_PARA_EXECUCAO (NFS e APS)\10_11_2026")    #PASTA ONDE FICAM AS NOTAS FISCAIS
PASTA_APS = Path(r"Y:\DOS_GOC_SCM\GOB\AUTOMAÇÃO - NÃO DELETAR\FILA_PARA_EXECUCAO (NFS e APS)\10_11_2026")    #PASTA ONDE FICAM AS APS

# LOGIN GOV
CPF = "x"
SENHA = "y"

#PERFIL NAVEGADOR (PRECISA MUDAR SE FOR EXECUTAR EM MAIS DE UM NAVEGADOR NO MESMO PC)
PERFIL_CHROME = r"C:\AutomacaoChrome\UserData1"

#BLOCO DE ASSINATURA 
bloco_inserir = "3780967"


#MARCADOR PROCESSO (DEPOIS DE FINALIZADO)
nome_marcardor = "AP - Aguardando Assinatura"


#colunas EXCEL
coluna_nome_NF = 1
Coluna_numero_NF = 2
Coluna_CONTRATO = 3
coluna_AP = 5
coluna_nome_AP = 6
Coluna_SEI = 7


# CONFIGURAÇÕES NAVEGADOR CHROME
URL = "https://minhaarea.sp.gov.br/plataformasp"
# ============================================================
# ABRIR CHROME E ACESSAR PLATAFORMA SP
# ============================================================

os.makedirs(PERFIL_CHROME, exist_ok=True)

options = Options()
options.add_argument(f"--user-data-dir={PERFIL_CHROME}")
options.add_argument("--profile-directory=Default")

navegador = webdriver.Chrome(options=options)
navegador.maximize_window()
navegador.get(URL)

# FECHAR OUTRAS GUIAS QUE POSSAM TER SIDO RESTAURADAS
aba_login = navegador.current_window_handle
for aba_chrome in navegador.window_handles:
    if aba_chrome != aba_login:
        navegador.switch_to.window(aba_chrome)
        navegador.close()

navegador.switch_to.window(aba_login)
#print("Chrome aberto na Plataforma SP!")

# ============================================================
# ABRIR TELA DE AUTENTICAÇÃO
# ============================================================

xpath_inicial = "/html/body/div/div/section/main/div[2]/div/sp-page-row/div/div/span[3]/div/div/div/div/div/div/div/div/div/a"
WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.XPATH, xpath_inicial))).click()

# ============================================================
# REALIZAR LOGIN SE NECESSÁRIO
# ============================================================

try:
    campo_cpf = WebDriverWait(navegador, 15).until(EC.element_to_be_clickable((By.XPATH, "/html/body/div[1]/main/form/div[1]/div[2]/input")))
    campo_cpf.send_keys(CPF)

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.XPATH, "/html/body/div[1]/main/form/div[1]/div[2]/div/button"))).click()

    campo_senha = WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.XPATH, "/html/body/div[1]/main/form/div[1]/div[1]/input")))
    campo_senha.send_keys(SENHA)

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.XPATH, "/html/body/div[1]/main/form/div[1]/div[2]/div/button[2]"))).click()
    print("Login enviado!")

except TimeoutException:
    print("Campo de CPF não encontrado. Prosseguindo para verificar a sessão.")

# ============================================================
# LOCALIZAR O CARTÃO DO SEI (SHADOW DOM)
# ============================================================

script_sei = """
function procurar(root) {
    for (const el of root.querySelectorAll('*')) {
        if (el.tagName?.toLowerCase() === 'now-card') {
            const div = (el.shadowRoot || el).querySelector('div');
            if (div) return div;
        }
        if (el.shadowRoot) {
            const encontrado = procurar(el.shadowRoot);
            if (encontrado) return encontrado;
        }
    }
    return null;
}
return procurar(document);
"""

elemento_sei = WebDriverWait(navegador, 90).until(
    lambda d: d.execute_script(script_sei)
)

div_sei = navegador.execute_script(
    "return (arguments[0].shadowRoot || arguments[0]).querySelector('div')",
    elemento_sei
)

if div_sei is None:
    raise RuntimeError("DIV interna do cartão SEI não encontrada.")

# CLICAR NO CARTÃO, COMO NO CÓDIGO ANTIGO
navegador.execute_script("arguments[0].scrollIntoView({block:'center'}); arguments[0].click();", div_sei)
print("Cartão do SEI clicado!")

# ============================================================
# AGUARDAR A GUIA DO SEI ABRIR
# ============================================================

def localizar_aba_sei(driver):
    for aba_chrome in driver.window_handles:
        driver.switch_to.window(aba_chrome)
        if "sei.sp.gov.br/sei/" in driver.current_url.lower():
            return aba_chrome
    return False

aba_sei = WebDriverWait(navegador, 90).until(localizar_aba_sei)
navegador.switch_to.window(aba_sei)

# CONFIRMAR QUE A INTERFACE DO SEI CARREGOU
WebDriverWait(navegador, 90).until(EC.presence_of_element_located((By.ID, "txtPesquisaRapida")))

# ============================================================
# FECHAR TODAS AS OUTRAS GUIAS
# ============================================================

for aba_chrome in navegador.window_handles:
    if aba_chrome != aba_sei:
        navegador.switch_to.window(aba_chrome)
        navegador.close()

navegador.switch_to.window(aba_sei)
navegador.switch_to.default_content()

print("Login concluído! Somente a guia do SEI permanece aberta.")
#############################################






##### EXECUÇÃO PRINCIPAL - LOOP ###############
ultima_linha = aba.max_row
while ultima_linha > 0 and aba[f'A{ultima_linha}'].value is None:
        ultima_linha -= 1

for i in range(LINHA_INICIAL,ultima_linha+1):

    os.system("cls")  # LIMPAR O TERMINAL
    navegador.switch_to.default_content() #COLOCA NO IFRAME PRINCIPAL
    print(f"LINHA EM EXECUÇÃO: {i}/{ultima_linha} | NF: {aba.cell(i, column=Coluna_numero_NF).value} | AP: {aba.cell(i, column=coluna_AP).value} | SEI: {aba.cell(i, column=Coluna_SEI).value}")


    processo = aba.cell(i, column=Coluna_SEI).value
    WebDriverWait(navegador, 90).until(EC.visibility_of_element_located((By.ID, 'txtPesquisaRapida'))).send_keys(processo)
    navegador.find_element(By.XPATH, '//*[@id="spnInfraUnidade"]/img').click()
    time.sleep(1)


    ID_IFRAME_CABECALHO_DOCUMENTOS = "ifrConteudoVisualizacao"
    WebDriverWait(navegador, 90).until(EC.frame_to_be_available_and_switch_to_it((By.ID, ID_IFRAME_CABECALHO_DOCUMENTOS)))
    time.sleep(0.2)





    # CLICAR EM INCLUIR DOCUMENTO (ATÉ 5 TENTATIVAS)
    ID_IFRAME_INSERIR_DOCUMENTOS = "ifrVisualizacao"
    for tentativa in range(1, 6):
        WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.XPATH, '//img[@title="Incluir Documento"]'))).click()
        time.sleep(2)

        try:
            navegador.switch_to.frame(WebDriverWait(navegador, 3).until(EC.presence_of_element_located((By.ID, ID_IFRAME_INSERIR_DOCUMENTOS))))
            WebDriverWait(navegador, 3).until(EC.element_to_be_clickable((By.ID, "txtFiltro")))
            #print("Campo para inserir nome do documento foi encontrado!")
            break
        except TimeoutException:
            print(f"Tentativa {tentativa}/5: campo txtFiltro não encontrado.")
            navegador.switch_to.parent_frame()
    else:
        raise RuntimeError("Campo txtFiltro não encontrado após 5 tentativas.")

    # VERIFICAR SE TODOS OS TIPOS ESTÃO SENDO EXIBIDOS
    if not navegador.find_elements(By.XPATH, '//img[@title="Exibir apenas os tipos já utilizados pela unidade"]'):
        WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "ancExibirSeries"))).click()
        print("Todos os tipos de documentos habilitados!")
    else:
        print("Todos os tipos de documentos já estavam habilitados!")

    time.sleep(0.5)
    # DIGITAR EXTERNO
    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "txtFiltro"))).send_keys("Externo")
    ##############




    ##
    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.XPATH, "//a[.//span[normalize-space()='Externo']]"))).click() #CLICA EM EXTERNO

    time.sleep(0.2)
    campo_tipo_do_documento = WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "selSerie")))
    Select(campo_tipo_do_documento).select_by_visible_text("Nota Fiscal")

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "txtDataElaboracao"))).send_keys(datetime.now().strftime("%d/%m/%Y"))  #insere data de hoje

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "txtNumero"))).send_keys(aba.cell(i, column=Coluna_numero_NF).value)  #insere número da nota fiscal 

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "txtNomeArvore"))).send_keys(aba.cell(i, column=Coluna_CONTRATO).value)  #insere nome na árvore 

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "divOptNato"))).click() #SELECIONAR OPÇÃO NATO-DIGITAL

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "divOptPublico"))).click() #SELECIONA NÍVEL DE ACESSO "PÚBLICO"

    nome_arquivo_NF = str(aba.cell(i, column=coluna_nome_NF).value).strip()
    caminho_arquivo_NF = PASTA_NFS / nome_arquivo_NF
    #caminho_arquivo_NF = PASTA_NFS / "TESTE ARQUIVO GRANDE.pdf"    #TESTE ARQUIVO GRANDE

    # VERIFICAR SE EXISTE O ARQUIVO
    if not caminho_arquivo_NF.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho_arquivo_NF}")

    WebDriverWait(navegador, 60).until(EC.presence_of_element_located((By.ID, "filArquivo"))).send_keys(str(caminho_arquivo_NF))  #anexar NF no SEI

    WebDriverWait(navegador, 90).until(EC.visibility_of_element_located((By.XPATH, '//img[@title="Remover Item"]')))  #aguarde até a NF for anexada

    time.sleep(0.3)
    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "btnSalvar"))).click()  #CLICA EM SALVAR

    time.sleep(0.6)
    try:
        alerta = WebDriverWait(navegador, 2).until(EC.alert_is_present())    #VERIFICAR SE APARECE A MSG "Já existe um documento (XX) cadastrado com estas características."
        mensagem = alerta.text
        #print("Alerta do SEI:", mensagem)
        alerta.accept()

    except TimeoutException:
        pass

    print("clicou em salvar")
    time.sleep(0.5)

    WebDriverWait(navegador, 120).until(EC.invisibility_of_element_located((By.ID, "btnSalvar")))         #AGUARDAR NF SER EFETIVAMENTE ESTAR ANEXADA NO SEI
    print("NF efetivamente anexada no SEI")
    ##





    #INSERIR AP

    time.sleep(1)
    navegador.switch_to.default_content()
    ID_IFRAME_CABECALHO_DOCUMENTOS = "ifrConteudoVisualizacao"
    WebDriverWait(navegador, 90).until(EC.frame_to_be_available_and_switch_to_it((By.ID, ID_IFRAME_CABECALHO_DOCUMENTOS)))
    time.sleep(0.5)





    ###
    # CLICAR EM INCLUIR DOCUMENTO (ATÉ 5 TENTATIVAS)

    for tentativa in range(1, 6):
        WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.XPATH, '//img[@title="Incluir Documento"]'))).click()
        time.sleep(2)

        try:
            navegador.switch_to.frame(WebDriverWait(navegador, 3).until(EC.presence_of_element_located((By.ID, ID_IFRAME_INSERIR_DOCUMENTOS))))
            WebDriverWait(navegador, 3).until(EC.element_to_be_clickable((By.ID, "txtFiltro")))
            print("Campo para inserir Autorização encontrado!")
            break
        except TimeoutException:
            print(f"Tentativa {tentativa}/5: campo txtFiltro não encontrado.")
            navegador.switch_to.parent_frame()
    else:
        raise RuntimeError("Campo txtFiltro não encontrado após 5 tentativas.")

    # VERIFICAR SE TODOS OS TIPOS ESTÃO SENDO EXIBIDOS
    if not navegador.find_elements(By.XPATH, '//img[@title="Exibir apenas os tipos já utilizados pela unidade"]'):
        WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "ancExibirSeries"))).click()
        print("Todos os tipos de documentos habilitados!")
    else:
        print("Todos os tipos de documentos já estavam habilitados!")

    # DIGITAR AUTORIZAÇÃO
    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "txtFiltro"))).send_keys("Autorização")
    #####



    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.XPATH, '//a[contains(@class,"ancoraOpcao") and normalize-space(.)="Autorização"]'))).click() #CLICA EM "Autorização"


    #TELA "GERAR DOCUMENTO"
    WebDriverWait(navegador, 90).until(EC.element_to_be_clickable((By.ID, "txtDescricao"))).send_keys(f"AP {aba.cell(i, column=coluna_AP).value}") #descrição

    WebDriverWait(navegador, 90).until(EC.element_to_be_clickable((By.ID, "txtNomeArvore"))).send_keys(f"AP {aba.cell(i, column=coluna_AP).value}") #nome na árvore


    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "divOptPublico"))).click() #SELECIONA NÍVEL DE ACESSO "PÚBLICO"


    aba_principal = navegador.current_window_handle  #salva a guia principal
    abas_antes = set(navegador.window_handles) # GUARDAR O NÚMERO DE ABAS EXISTENTES ANTES DE SALVAR

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "btnSalvar"))).click()  #CLICA EM SALVAR
    time.sleep(0.2)

    WebDriverWait(navegador, 90).until(lambda d: len(set(d.window_handles) - abas_antes) > 0)  #IDENTIFICA QUANDO É ABERTO NOVA ABA ("INSERIR DESPACHO AP" NESSE CASO)

    nova_aba = next(iter(set(navegador.window_handles) - abas_antes)) #Identificar qual é a nova ABA

    navegador.switch_to.window(nova_aba) #MUDA PARA NOVA ABA

    print("mudou para nova ABA para inserir AP")
    time.sleep(0.2)

    WebDriverWait(navegador, 60).until(EC.frame_to_be_available_and_switch_to_it((By.CSS_SELECTOR, 'iframe.cke_wysiwyg_frame[title="Corpo do Texto"]'))) # ENTRAR NO IFRAME DO EDITOR

    campo_texto = WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'body[contenteditable="true"]'))) # LOCALIZAR O CORPO EDITÁVEL

    # CLICAR, SELECIONAR TUDO, APAGAR E ESCREVER
    campo_texto.click()
    campo_texto.send_keys(Keys.CONTROL, "a")
    campo_texto.send_keys(Keys.BACKSPACE)
    print("limpou o corpo do texto")
    time.sleep(0.3)






    navegador.switch_to.default_content() #Volta para o iframe principal
    WebDriverWait(navegador, 60).until(lambda d: next((e for e in d.find_elements(By.CSS_SELECTOR, 'a.cke_button__base64image[title="Imagem"]') if e.is_displayed() and e.is_enabled() and e.get_attribute("aria-disabled") == "false"), None)).click() #CLICA NA IMAGEM (NO CABECALHO), PARA INSERIR ARQUIVOS
    print("clicou no símbolo de imagem, no cabeçalho, para inserir arquivo para anexar")

    time.sleep(0.2)
    WebDriverWait(navegador, 60).until(EC.visibility_of_element_located((By.CSS_SELECTOR,"table.cke_dialog")))   #AGUARDAR TELA DE INSERIR ARQUIVO
    print("tela de inserir arquivo aparecer")


    WebDriverWait(navegador, 90).until(EC.frame_to_be_available_and_switch_to_it((By.CSS_SELECTOR, "iframe.cke_dialog_ui_input_file")))
    print("mudou para iframe inserir selecionar arquivo do pc")


    nome_arquivo_AP = str(aba.cell(i, column=coluna_nome_AP).value).strip()
    caminho_arquivo_AP = None
    for arquivo in PASTA_APS.iterdir():
        if arquivo.stem == nome_arquivo_AP and arquivo.suffix.lower() in (".jpg", ".jpeg", ".png"):   #procurar por essas extensões que são aceitas no SEI
            caminho_arquivo_AP = arquivo
            break

    # 3. VERIFICAR SE ENCONTROU
    if caminho_arquivo_AP is None:
        raise FileNotFoundError(f"Imagem não encontrada: {nome_arquivo_AP}")


    WebDriverWait(navegador, 60).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'input[type="file"][name="filArquivo"]'))).send_keys(str(caminho_arquivo_AP)) #INSERE AP


    navegador.switch_to.default_content() #Volta para o iframe principal
    time.sleep(0.4)

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a.cke_dialog_ui_button_ok"))).click()  #clica em OK" para inserir o ARQUIVO efetivamente
    WebDriverWait(navegador, 60).until(EC.invisibility_of_element_located((By.CSS_SELECTOR, "table.cke_dialog"))) # AGUARDAR A JANELA DE IMAGEM FECHAR
    time.sleep(0.3)

    WebDriverWait(navegador, 60).until(EC.frame_to_be_available_and_switch_to_it((By.CSS_SELECTOR, "iframe.cke_wysiwyg_frame"))) # ENTRAR NO IFRAME DO TEXTO
    WebDriverWait(navegador, 90).until(EC.presence_of_element_located((By.CSS_SELECTOR, "body img")))   #aguarda pelo menos uma imagem aparecer
    print("apareceu a imagem na tela")
    time.sleep(0.2)

    navegador.switch_to.default_content() # VOLTAR PARA A PÁGINA DO EDITOR

    ActionChains(navegador)\
        .key_down(Keys.CONTROL)\
        .key_down(Keys.ALT)\
        .send_keys("s")\
        .key_up(Keys.ALT)\
        .key_up(Keys.CONTROL)\
        .perform()   #clicar em salvar usando atalho (ctrl + alt + s)
    print("salvou por meio do atalho ctrl + alt + s")

    WebDriverWait(navegador, 90).until(lambda d: d.find_element(By.CSS_SELECTOR, "a.cke_button__save").get_attribute("aria-disabled") == "true")  #AGUARDAR O BOTÃO FICAR DESABILITADO
    time.sleep(0.3)
    print("Botão Salvar desabilitado! Salvamento concluído.")
    navegador.close()  # Fecha a guia atual (editor da AP)
    ######




    ###3. INCLUINDO NO BLOCO
    navegador.switch_to.window(aba_principal)
    navegador.switch_to.default_content()
    print("Editor da AP fechado. Retornou ao SEI principal!")

    WebDriverWait(navegador, 90).until(EC.frame_to_be_available_and_switch_to_it((By.ID, ID_IFRAME_CABECALHO_DOCUMENTOS)))
    time.sleep(0.2)

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.XPATH, '//img[@title="Incluir em Bloco de Assinatura"]'))).click()

    WebDriverWait(navegador, 90).until(EC.presence_of_element_located((By.ID, "ifrVisualizacao")))  #AGUARDAR IFRAME
    navegador.switch_to.frame(navegador.find_element(By.ID, "ifrVisualizacao"))  #mudar para iframe
    time.sleep(0.2)

    WebDriverWait(navegador, 60).until(EC.presence_of_element_located((By.ID, "selBloco"))) #SELECIONAR BLOCO PELO NÚMERO
    Select(navegador.find_element(By.ID, "selBloco")).select_by_value(bloco_inserir)
    time.sleep(0.3)

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "sbmIncluir"))).click()     #INCLUIR NO BLOCO
    time.sleep(0.5)

    WebDriverWait(navegador, 20).until(EC.presence_of_element_located((By.XPATH,f'//*[@id="tblDocumentos"]//tr[td][last()]/td[5]/a[normalize-space()="{bloco_inserir}"]'))) #aguarda o bloco inserir efetivamente
    print("AP inserida no bloco")
    time.sleep(0.3)




    #inserindo TAG

    navegador.switch_to.default_content()
    WebDriverWait(navegador, 90).until(EC.presence_of_element_located((By.ID, "ifrArvore")))  #AGUARDAR IFRAME
    navegador.switch_to.frame(navegador.find_element(By.ID, "ifrArvore"))
    print("entrou no iframe para clicar no SEI")
    time.sleep(0.2)
    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.infraArvoreNo[href*="acao=arvore_visualizar"][href*="id_procedimento="]'))).click()   #clica no número do processo
    print("SEI CLICADO PARA INSERIR TAG")
    time.sleep(0.8)

    navegador.switch_to.default_content()
    ID_IFRAME_CABECALHO_DOCUMENTOS = "ifrConteudoVisualizacao"
    WebDriverWait(navegador, 90).until(EC.presence_of_element_located((By.ID, ID_IFRAME_CABECALHO_DOCUMENTOS)))  #AGUARDAR IFRAME DE CABEÇALHO DE DOCUMENTOS
    navegador.switch_to.frame(navegador.find_element(By.ID, ID_IFRAME_CABECALHO_DOCUMENTOS))
    print("iframe carregado")

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.XPATH, '//img[@title="Gerenciar Marcador"]'))).click()  #clicar em inserir TAG
    print("clicou na tag")
    time.sleep(0.3)

    # ENTRAR NO IFRAME
    WebDriverWait(navegador, 90).until(EC.frame_to_be_available_and_switch_to_it((By.ID, "ifrVisualizacao")))
    time.sleep(0.1)

    # VERIFICAR SE EXISTE O BOTÃO ADICIONAR
    if navegador.find_elements(By.ID, "btnAdicionar"):
        WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "btnAdicionar"))).click()
        print("Botão Adicionar encontrado e clicado!")

        # AGUARDAR TELA DE CADASTRO DO MARCADOR
        WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.dd-select span.dd-pointer")))
    else:
        print("Botão Adicionar não encontrado. Selecionando marcador diretamente.")


    # SELECIONAR MARCADOR
    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.dd-select span.dd-pointer"))).click()

    time.sleep(0.1)
    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.XPATH, f'//label[contains(@class,"dd-option-text") and normalize-space()="{nome_marcardor}"]'))).click()
    time.sleep(0.4)

    WebDriverWait(navegador, 60).until(EC.element_to_be_clickable((By.ID, "sbmSalvar"))).click() 
    time.sleep(1)
    

print("TODOS OS PROCESSOS FINALIZADOS!")  

 

1 - Página de login / autenticação aberta
Verificando se existe sessão autenticada...
Sessão não encontrada.
Login será realizado.
2 - CPF preenchido
3 - Botão continuar clicado
4 - Senha preenchida
5 - Login enviado
6 - Aguardando 20 segundos para carregar a aplicação...
7 - Procurando o elemento...
8 - NOW-CARD encontrado
9 - DIV interna encontrada
10 - CLIQUE FINAL REALIZADO COM SUCESSO!

11 - Aguardando janela / aba do SEI...
Verificando: Login Minha Área SP - Plataforma SP | https://minhaarea.sp.gov.br/plataformasp
Verificando: Página inicial | Minha Área SP.GOV.BR | ServiceNow | https://plataforma.sp.gov.br/x/mdpdd/minha-area-2-0/minha-area-2-0
Verificando: SEI - Controle de Processos | https://sei.sp.gov.br/sei/controlador.php?acao=procedimento_controlar&acao_origem=principal&acao_retorno=principal&inicializando=1&infra_sistema=100000100&infra_unidade_atual=110000700&infra_hash=e4f2192165d3ec9e8d0a32475e0e9c2a61612229d362aa55d407da2d2538e3b2

12 - Aba do SEI localizada.
Título: 